In [1]:
import touchterrain.common.dem_tile_export as dte
from pathlib import Path

In [2]:

# ------------------------------------------------------------
# 1 ▸ Define AOI (pentagon in San Gabriel Mountains)
# ------------------------------------------------------------
pentagon_geojson = {
    "type": "Polygon",
    "coordinates": [[
        [-118.175, 34.28],
        [-118.140, 34.26],
        [-118.120, 34.23],
        [-118.090, 34.26],
        [-118.110, 34.29],
        [-118.175, 34.28]  # closed ring
    ]]
}

# ------------------------------------------------------------
# 2 ▸ Fetch DEM with 250 m fringe
# ------------------------------------------------------------
dem_np, dem_img, bb = dte.fetch_dem_with_fringe(
    geojson=pentagon_geojson,
    scale=30,
    fringe_m=250
)

# ------------------------------------------------------------
# 3 ▸ Convert DEM to surface mesh and save STL
# ------------------------------------------------------------
mesh = dte.dem_to_surface_stl(
    dem_np=dem_np,
    bbox=bb,
    out_stl="pentagon_dem_surface.stl",
    vertical_exaggeration=1.2,
    show_k3d=True
)

# ------------------------------------------------------------
# 4 ▸ Create watertight block from surface mesh
# ------------------------------------------------------------
rows, cols = dem_np.shape
dem_block = dte.dem_sheet_to_block(
    surf_mesh=mesh,
    rows=rows,
    cols=cols,
    base_value=5.0,
    base_mode="absolute"
)

# Save block to STL (optional)
block_path = Path("watertight_dem_block.stl")
dem_block.export(block_path)
print(f"✓ Saved: {block_path}")

# ------------------------------------------------------------
# 5 ▸ Clip with AOI prism (reprojected & scaled)
# ------------------------------------------------------------
clipped = dte.clip_dem_block(
    dem_block=dem_block,
    aoi_geojson=pentagon_geojson,
    out_stl="dem_clipped.stl",
    prism_stl="aoi_prism.stl",
    z_padding=20.0,
    project_crs="auto",       # or set manually e.g. "EPSG:32611"
    xy_scale=1/1000,          # convert to millimeters
    z_scale=2/1000,           # 2× vertical exaggeration in mm
    show_k3d=True
)

✓ STL written to /TouchTerrain/standalone/pentagon_dem_surface.stl


/opt/conda/lib/python3.10/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float32" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "uint32" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

✓ Saved: watertight_dem_block.stl
✓ AOI prism STL saved: /TouchTerrain/standalone/aoi_prism.stl
⧗ Boolean intersection …
✓ Clipped STL saved: /TouchTerrain/standalone/dem_clipped.stl


Output()